### Solving the Optimization Problem with Julia

Mixed-integer facility location model (p-median) for choosing **P = 5** ambulance standby sites.

$$
\min_{x,y}\ \frac{\sum_{i\in I}\sum_{j\in J} w_i\, t_{ji}\, x_{ji}}{\sum_{i\in I} w_i}
\quad \text{s.t.}\quad \sum_{j\in J} y_j = P,\qquad \sum_{j\in J} x_{ji} = 1\ \ (\forall i\in I),\qquad x_{ji}\le y_j\ \ (\forall i\in I,\ j\in J),\qquad x,y\in\{0,1\}
$$

- $I$: ZIP areas (demand points), $w_i$: call count in area $i$ during the planning period
- $J$: candidate standby sites, $t_{ji}$: estimated driving time (minutes) from site $j$ to area $i$'s representative point
- $y_j$: site $j$ is selected; $x_{ji}$: site $j$ serves area $i$

Data files:
- `data/demand.csv`: one row per Brooklyn ZIP area: `area_id, lat, lon, calls` (the area's representative point and its weekday-daytime call count, built in `Analysis.ipynb` / `Geography.ipynb`; ZIP 11385 already excluded). `data/demand_centroids.csv` has the same schema but uses polygon centroids instead.
- `data/candidate_sites.csv`: candidate standby sites $J$ (the 66 Brooklyn FDNY firehouses) with their `site_id`s. Built by `TravelTimes.ipynb`.
- `data/travel_times.csv`: $t_{ji}$ in long format `site_id, area_id, euclid_miles, network_miles, minutes` (shortest street-network route at a uniform speed). Built by `TravelTimes.ipynb`; run that notebook first.

In [ ]:
using CSV, DataFrames, JuMP, Gurobi, Printf

#### 1. Configuration

In [ ]:
DATA_DIR   = "data"
OUTPUT_DIR = "results"

DEMAND_FILE      = joinpath(DATA_DIR, "demand.csv")                          # area_id, lat, lon, calls
CANDIDATE_FILE   = joinpath(DATA_DIR, "candidate_sites.csv")                 # site_id, name, address, lat, lon
TRAVEL_TIME_FILE = joinpath(DATA_DIR, "travel_times.csv")                    # site_id, area_id, ..., minutes

P          = 5               # number of standby sites to open
WEIGHT_COL = :calls          # w_i: call count per ZIP area
TIME_COL   = :minutes        # t_ji column in TRAVEL_TIME_FILE

TIME_LIMIT_SEC = 300

#### 2. Load demand areas ($I$, $w_i$)

In [ ]:
demand = CSV.read(DEMAND_FILE, DataFrame)
demand.area_id = string.(demand.area_id)
dropmissing!(demand, [:lat, :lon, WEIGHT_COL])
filter!(row -> row[WEIGHT_COL] > 0, demand)

I_areas = demand.area_id
w = Float64.(demand[!, WEIGHT_COL])
println("Demand areas |I| = $(length(I_areas)),  total $(WEIGHT_COL) = $(sum(w))")
first(demand, 5)

#### 3. Load candidate standby sites ($J$)

In [ ]:
candidates = CSV.read(CANDIDATE_FILE, DataFrame)
candidates.site_id = string.(candidates.site_id)

J_sites = candidates.site_id
println("Candidate sites |J| = $(length(J_sites))")
first(candidates, 5)

#### 4. Travel times $t_{ji}$

`t[j, i]` = minutes from site `j` to area `i`, read from `TRAVEL_TIME_FILE` (built by `TravelTimes.ipynb`).

In [ ]:
function load_travel_times(path, J_sites, I_areas; col=TIME_COL)
    tt = CSV.read(path, DataFrame)
    tt.site_id = string.(tt.site_id)
    tt.area_id = string.(tt.area_id)
    jidx = Dict(s => k for (k, s) in enumerate(J_sites))
    iidx = Dict(a => k for (k, a) in enumerate(I_areas))
    t = fill(NaN, length(J_sites), length(I_areas))
    for r in eachrow(tt)
        haskey(jidx, r.site_id) && haskey(iidx, r.area_id) || continue
        t[jidx[r.site_id], iidx[r.area_id]] = r[col]
    end
    n_missing = count(isnan, t)
    n_missing == 0 || error("$(n_missing) (site, area) pairs missing from $(path): rerun TravelTimes.ipynb")
    return t
end

t = load_travel_times(TRAVEL_TIME_FILE, J_sites, I_areas)

@printf("t_ji: %d × %d, range %.1f–%.1f min\n", size(t)..., minimum(t), maximum(t))

#### 5. Build and solve the MIP

In [ ]:
function solve_standby_model(t, w, P; time_limit=TIME_LIMIT_SEC, silent=true)
    nJ, nI = size(t)
    model = Model(Gurobi.Optimizer)
    silent && set_silent(model)
    set_time_limit_sec(model, time_limit)

    @variable(model, y[1:nJ], Bin)          # site j selected
    @variable(model, x[1:nJ, 1:nI], Bin)    # site j serves area i

    @objective(model, Min, sum(w[i] * t[j, i] * x[j, i] for j in 1:nJ, i in 1:nI) / sum(w))

    @constraint(model, open_P, sum(y) == P)
    @constraint(model, assign[i in 1:nI], sum(x[j, i] for j in 1:nJ) == 1)
    @constraint(model, link[j in 1:nJ, i in 1:nI], x[j, i] <= y[j])

    optimize!(model)
    is_solved_and_feasible(model) || error("Solve failed: $(termination_status(model))")

    selected = findall(>(0.5), value.(y))
    assigned = [findfirst(>(0.5), value.(x[:, i])) for i in 1:nI]
    return (model=model, selected=selected, assigned=assigned, objective=objective_value(model))
end

sol = solve_standby_model(t, w, P)
println("Status: ", termination_status(sol.model), " | solve time: ",
        round(solve_time(sol.model), digits=2), " s")
@printf("Call-weighted average travel time: %.2f min\n", sol.objective)

#### 6. Results

In [ ]:
selected_sites = candidates[sol.selected, :]
selected_sites.n_areas_served = [count(==(j), sol.assigned) for j in sol.selected]
selected_sites.calls_served   = [sum(w[sol.assigned .== j]) for j in sol.selected]
selected_sites

In [ ]:
assignment = DataFrame(
    area_id   = I_areas,
    weight    = w,
    site_id   = J_sites[sol.assigned],
    site_name = candidates.name[sol.assigned],
    minutes   = [t[sol.assigned[i], i] for i in eachindex(I_areas)],
)
sort(assignment, :minutes, rev=true)

#### 7. Save outputs

In [ ]:
mkpath(OUTPUT_DIR)
CSV.write(joinpath(OUTPUT_DIR, "selected_sites_P$(P).csv"), selected_sites)
CSV.write(joinpath(OUTPUT_DIR, "area_assignment_P$(P).csv"), assignment)
println("Wrote results to $(OUTPUT_DIR)/")